In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

# 1. Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 2. Hyperparameters
num_epochs = 10
batch_size = 64
learning_rate = 0.001

# 3. Data preprocessing and loading
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5),  # mean for RGB
                         (0.5, 0.5, 0.5))  # std for RGB
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                             download=True, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                            download=True, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# 4. Define CNN model
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.fc1 = nn.Linear(64 * 8 * 8, 512)
        self.fc2 = nn.Linear(512, 10)

    def forward(self, x):
        x = self.pool(torch.relu(self.conv1(x)))   # [B, 32, 16, 16]
        x = self.pool(torch.relu(self.conv2(x)))   # [B, 64, 8, 8]
        x = x.view(-1, 64 * 8 * 8)                 # Flatten
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = CNN().to(device)

# 5. Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# 6. Training loop
for epoch in range(num_epochs):
    running_loss = 0.0
    for i, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if (i + 1) % 100 == 0:
            print(f"Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], Loss: {loss.item():.4f}")

# 7. Evaluation
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Test Accuracy: {100 * correct / total:.2f}%")


100%|██████████| 170M/170M [00:16<00:00, 10.3MB/s]


Epoch [1/10], Step [100/782], Loss: 1.7300
Epoch [1/10], Step [200/782], Loss: 1.3319
Epoch [1/10], Step [300/782], Loss: 1.3860
Epoch [1/10], Step [400/782], Loss: 1.2715
Epoch [1/10], Step [500/782], Loss: 1.4546
Epoch [1/10], Step [600/782], Loss: 0.7545
Epoch [1/10], Step [700/782], Loss: 1.2314
Epoch [2/10], Step [100/782], Loss: 1.0342
Epoch [2/10], Step [200/782], Loss: 0.9279
Epoch [2/10], Step [300/782], Loss: 0.6899
Epoch [2/10], Step [400/782], Loss: 0.8078
Epoch [2/10], Step [500/782], Loss: 0.7099
Epoch [2/10], Step [600/782], Loss: 1.0406
Epoch [2/10], Step [700/782], Loss: 0.9383
Epoch [3/10], Step [100/782], Loss: 0.7776
Epoch [3/10], Step [200/782], Loss: 0.7889
Epoch [3/10], Step [300/782], Loss: 0.6018
Epoch [3/10], Step [400/782], Loss: 0.6752
Epoch [3/10], Step [500/782], Loss: 0.6888
Epoch [3/10], Step [600/782], Loss: 0.6499
Epoch [3/10], Step [700/782], Loss: 0.6753
Epoch [4/10], Step [100/782], Loss: 0.4966
Epoch [4/10], Step [200/782], Loss: 0.3932
Epoch [4/10